# 04 · Models and validation: baseline ladder, paired comparisons, SHAP, conformal
* Hyper-parameters are tuned **once** on the fixed tuning split (train <= 2013,
  valid 2014) and then frozen.
* Walk-forward validation on 2015-2023: one model per year, expanding window,
  22-session purge. Holdout rows do not exist in this notebook.

In [1]:
import json
import warnings
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import shap

from etfmom.config import ART, FIG, SEED, TAB, setup
from etfmom.features import STATES
from etfmom.ledger import log_run
from etfmom.metrics import daily_ic, ic_summary, nw_mean
from etfmom.models import (LABEL, LGBMModel, RidgeModel, RuleModel, ensemble, make_target, selection_rows, train_rows,
                           tuning_fold, valid_rows, validation_folds)

plt.style.use("seaborn-v0_8-whitegrid")
S = setup()
panel = pd.read_parquet(ART / "panel_dev.parquet")
sel = json.loads((ART / "selected_features.json").read_text())
SIGNALS = sel["selected_signals"]
FEATS = SIGNALS + STATES
sessions = panel.index.get_level_values("date").unique().sort_values()
print("signals:", SIGNALS); print("states:", STATES)

signals: ['mom_252_21', 'mom_126_21', 'mom_63', 'sharpe_126', 'sma_ratio_200', 'trend_consistency_63', 'mom_accel', 'skew_63', 'amihud_63']
states: ['asset_rvol_pct', 'mkt_rvol_pct', 'mkt_trend', 'corr_spy_tlt_63', 'xs_disp_21']


## Tuning split: Ridge alpha sweep (10 values, 9 orders of magnitude)

In [2]:
tf = tuning_fold(sessions)
print(tf)
tr = train_rows(panel, tf, sessions)
va = selection_rows(panel, tf.valid_start, f"{S['evaluation']['validation_folds'][0]}-01-01", sessions)  # purged at the 2015 boundary
print("tuning valid:", va.index.get_level_values("date").min().date(), "..", va.index.get_level_values("date").max().date())
alpha_rows = []
for a in S["models"]["ladder"][1]["alpha_grid"]:
    m = RidgeModel(SIGNALS, a).fit(tr)
    ic = daily_ic(va.assign(pred=m.predict(va)), "pred", LABEL)
    s = ic_summary(ic)
    alpha_rows.append({"alpha": a, "ic": s["mean"], "icir": s["icir_daily"]})
    log_run("ridge_alpha", f"alpha={a:g}", "tuning_2014", {"alpha": a, "features": SIGNALS}, "valid 2014", {"ic": s["mean"], "icir": s["icir_daily"]})
alpha_tab = pd.DataFrame(alpha_rows)
best_alpha = float(alpha_tab.loc[alpha_tab.ic.idxmax(), "alpha"])
alpha_tab.to_csv(TAB / "04_ridge_alpha.csv", index=False)
print("chosen alpha:", best_alpha); alpha_tab.round(4)

Fold(name='tuning_2014', train_start=Timestamp('2007-01-01 00:00:00'), train_end=Timestamp('2013-11-27 00:00:00'), valid_start=Timestamp('2014-01-02 00:00:00'), valid_end=Timestamp('2014-12-31 00:00:00'))
tuning valid: 2014-01-02 .. 2014-11-28


chosen alpha: 100000.0


,alpha,ic,icir
0,0.0001,0.0647,0.2051
1,0.0010,0.0647,0.2051
2,0.0100,0.0647,0.2051
3,0.1000,0.0647,0.2051
4,1.0000,0.0647,0.2051
5,10.0000,0.0647,0.2050
6,100.0000,0.0646,0.2050
7,1000.0000,0.0645,0.2053
8,10000.0000,0.0656,0.2186
9,100000.0000,0.0709,0.2501


## Tuning split: LightGBM, Optuna TPE, 50 trials, early stopping on 2014

In [3]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    p = {"num_leaves": trial.suggest_int("num_leaves", 4, 31),
         "max_depth": trial.suggest_int("max_depth", 2, 5),
         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
         "min_child_samples": trial.suggest_int("min_child_samples", 100, 2000, log=True),
         "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
         "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0), "bagging_freq": 1,
         "lambda_l2": trial.suggest_float("lambda_l2", 1e-3, 10, log=True)}
    m = LGBMModel(FEATS, p, 2000).fit(tr, va, early_stopping=100)
    ic = daily_ic(va.assign(pred=m.predict(va)), "pred", LABEL).mean()
    trial.set_user_attr("best_iteration", int(m.best_iteration))
    log_run("lgbm_optuna", f"trial_{trial.number}", "tuning_2014", {**p, "best_iteration": int(m.best_iteration)}, "valid 2014", {"ic": float(ic)})
    return ic

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=S["models"]["ladder"][2]["max_optuna_trials"])
best = study.best_trial
LGB_PARAMS = {**best.params, "bagging_freq": 1}
LGB_ROUNDS = int(best.user_attrs["best_iteration"])
print("best IC 2014:", round(best.value, 4), "| rounds:", LGB_ROUNDS, "|", LGB_PARAMS)
trials = study.trials_dataframe()[["number", "value", "user_attrs_best_iteration"]]
trials.to_csv(TAB / "04_optuna_trials.csv", index=False)
json.dump({"ridge_alpha": best_alpha, "lgbm_params": LGB_PARAMS, "lgbm_rounds": LGB_ROUNDS, "signals": SIGNALS, "states": STATES},
          open(ART / "frozen_hyperparams.json", "w"), indent=2)

best IC 2014: 0.1411 | rounds: 295 | {'num_leaves': 12, 'max_depth': 3, 'learning_rate': 0.011641142127508471, 'min_child_samples': 562, 'feature_fraction': 0.9278790092288136, 'bagging_fraction': 0.9307466578040249, 'lambda_l2': 1.841177172308202, 'bagging_freq': 1}


## Walk-forward validation 2015-2023 (frozen hyper-parameters)

In [4]:
folds = validation_folds(sessions)
fold_tab = pd.DataFrame([vars(f) for f in folds])
fold_tab.to_csv(TAB / "04_fold_dates.csv", index=False)
preds, shap_store, conformal_rows, ridge_coefs = [], {}, [], {}
for f in folds:
    tr, va = train_rows(panel, f, sessions), valid_rows(panel, f)
    assert tr.index.get_level_values("date").max() <= f.train_end < f.valid_start
    out = va[[LABEL]].copy()
    rule = RuleModel().fit(tr); out["rule"] = rule.predict(va)
    rdg = RidgeModel(SIGNALS, best_alpha).fit(tr); out["ridge"] = rdg.predict(va); ridge_coefs[f.name] = rdg.coef_
    gbm = LGBMModel(FEATS, LGB_PARAMS, LGB_ROUNDS).fit(tr); out["lgbm"] = gbm.predict(va)
    out["ensemble"] = ensemble({"rule": out["rule"], "ridge": out["ridge"], "lgbm": out["lgbm"]})
    out["fold"] = f.name
    preds.append(out)
    # SHAP on a fixed-size sample of validation rows
    samp = va[FEATS].sample(min(3000, len(va)), random_state=SEED)
    sv = shap.TreeExplainer(gbm.m).shap_values(samp)
    shap_store[f.name] = (samp, sv)
    # split conformal: fit on core, calibrate on the last 252 training sessions (purged from core)
    tr_dates = tr.index.get_level_values("date")
    cal_start = sessions[sessions.get_loc(f.train_end) - 252]
    core_end = sessions[sessions.get_loc(cal_start) - S["evaluation"]["label_buffer_sessions"] - 1]
    core, cal = tr[tr_dates <= core_end], tr[tr_dates >= cal_start]
    for name, mk in (("ridge", lambda: RidgeModel(SIGNALS, best_alpha)), ("lgbm", lambda: LGBMModel(FEATS, LGB_PARAMS, LGB_ROUNDS))):
        m = mk().fit(core)
        yc, _ = make_target(cal, m.clip); res = (yc - m.predict(cal)).abs()
        yv, _ = make_target(va.dropna(subset=[LABEL]), m.clip); pv = m.predict(va.dropna(subset=[LABEL]))
        for lvl in (0.8, 0.9):
            q = res.quantile(lvl)
            conformal_rows.append({"fold": f.name, "model": name, "level": lvl, "coverage": float(((yv - pv).abs() <= q).mean()), "half_width": float(q)})
    print(f.name, "train rows", len(tr), "| valid rows", len(va))
pred = pd.concat(preds)
pred.to_parquet(ART / "validation_predictions.parquet")
for k in ("rule", "ridge", "lgbm", "ensemble"):
    log_run("model_ladder", k, "walk_forward_2015_2023", {"alpha": best_alpha} if k == "ridge" else ({"params": LGB_PARAMS, "rounds": LGB_ROUNDS} if k == "lgbm" else {}),
            "2015-2023", {"ic": float(daily_ic(pred, k, LABEL).mean())})

2015 train rows 28714 | valid rows 21944


2016 train rows 33097 | valid rows 22845


2017 train rows 37613 | valid rows 23285


2018 train rows 42307 | valid rows 23536


2019 train rows 47015 | valid rows 23643


2020 train rows 51713 | valid rows 23662


2021 train rows 56472 | valid rows 23916


2022 train rows 61226 | valid rows 23720


2023 train rows 65970 | valid rows 22816


## Yardstick: mean daily Spearman IC with Newey-West 95% CI (lag 21)

In [5]:
ics = {k: daily_ic(pred, k, LABEL) for k in ("rule", "ridge", "lgbm", "ensemble")}
summ = pd.DataFrame({k: ic_summary(v) for k, v in ics.items()}).T[["mean", "lo", "hi", "p", "icir_daily", "share_pos_folds", "n_eff", "T"]]
summ.columns = ["IC", "CI_lo", "CI_hi", "p", "ICIR(daily)", "share_pos_folds", "n_eff_days", "days"]
summ.to_csv(TAB / "04_ic_summary.csv")
summ.round(4)

,IC,CI_lo,CI_hi,p,ICIR(daily),share_pos_folds,n_eff_days,days
rule,0.0187,-0.0288,0.0663,0.4394,0.0576,0.6667,128.4123,2242.0
ridge,0.0450,0.0054,0.0846,0.0260,0.1591,0.7778,142.1176,2242.0
lgbm,0.0776,0.0418,0.1134,0.0000,0.2998,0.7778,147.2078,2242.0
ensemble,0.0603,0.0197,0.1009,0.0036,0.2103,0.6667,137.2118,2242.0


In [6]:
per_fold = pd.DataFrame({k: v.groupby(v.index.year).mean() for k, v in ics.items()})
per_fold_ci = {k: pd.DataFrame({y: nw_mean(v[v.index.year == y]) for y in per_fold.index}).T for k, v in ics.items()}
fig, ax = plt.subplots(figsize=(9, 3.4))
for i, k in enumerate(per_fold.columns):
    c = per_fold_ci[k]
    ax.errorbar(per_fold.index + (i - 1.5) * 0.15, c["mean"], yerr=[c["mean"] - c["lo"], c["hi"] - c["mean"]], fmt="o", capsize=2, label=k)
ax.axhline(0, color="k", lw=0.8)
ax.set_title("Q: is the signal stable fold by fold? (mean daily IC, NW 95% band)"); ax.set_ylabel("Spearman IC"); ax.legend(ncol=4, fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "04_ic_per_fold.png", dpi=150)
per_fold.round(4).to_csv(TAB / "04_ic_per_fold.csv"); per_fold.round(4)

,rule,ridge,lgbm,ensemble
date,,,,
2015,0.1284,0.0453,0.1171,0.1037
2016,-0.1628,-0.1161,-0.0299,-0.1343
2017,0.1823,0.1577,0.1814,0.1851
2018,-0.1037,-0.0480,0.0295,-0.0566
2019,0.0525,0.0435,0.1398,0.0945
2020,0.0111,0.1457,0.1304,0.1656
2021,0.0313,0.1255,0.1419,0.1221
2022,0.0426,0.0358,0.0140,0.0680
2023,-0.0161,0.0120,-0.0366,-0.0130


## Paired comparisons up the ladder (daily IC difference, NW 95% CI)

In [7]:
def paired(a, b):
    d = (ics[a] - ics[b]).dropna(); s = nw_mean(d)
    return {"comparison": f"{a} - {b}", "dIC": s["mean"], "lo": s["lo"], "hi": s["hi"], "p": s["p"]}
pairs = pd.DataFrame([paired("ridge", "rule"), paired("lgbm", "ridge"), paired("lgbm", "rule"), paired("ensemble", "rule"), paired("ensemble", "ridge"), paired("ensemble", "lgbm")])
pairs.round(4).to_csv(TAB / "04_paired.csv", index=False)
# pre-registered deploy rule (setup.yaml models.deploy_rule): deploy the MOST COMPLEX rung whose paired
# daily-IC difference against the BEST simpler rung (highest validation mean IC) has NW 95% lower bound > 0
LADDER = ["rule", "ridge", "lgbm", "ensemble"]
deployed, rule_trace = "rule", []
for k in reversed(LADDER[1:]):
    simpler = LADDER[:LADDER.index(k)]
    best_simpler = max(simpler, key=lambda s: ics[s].mean())
    lo = paired(k, best_simpler)["lo"]
    rule_trace.append(f"{k} vs best simpler {best_simpler}: lo={lo:.4f}")
    if lo > 0:
        deployed = k
        break
print("\n".join(rule_trace)); print("deployed rung:", deployed)
json.dump({"deployed": deployed}, open(ART / "deployed.json", "w"))
log_run("model_ladder", "deploy_decision", "walk_forward_2015_2023", {"rule": S["models"]["deploy_rule"]}, "2015-2023",
        {**{r.comparison: float(r.dIC) for r in pairs.itertuples()}, "trace": rule_trace}, f"deploy={deployed}", "paired NW CI lower bound > 0 vs best simpler rung")
pairs.round(4)

ensemble vs best simpler lgbm: lo=-0.0422
lgbm vs best simpler ridge: lo=0.0026
deployed rung: lgbm


,comparison,dIC,lo,hi,p
0,ridge - rule,0.0262,-0.0199,0.0724,0.2651
1,lgbm - ridge,0.0326,0.0026,0.0627,0.0331
2,lgbm - rule,0.0589,0.0112,0.1066,0.0156
3,ensemble - rule,0.0415,0.0084,0.0747,0.0141
4,ensemble - ridge,0.0153,-0.0053,0.0359,0.1462
5,ensemble - lgbm,-0.0173,-0.0422,0.0075,0.1706


## SHAP (LightGBM): magnitude, sign consistency, stability across folds

In [8]:
imp = pd.DataFrame({k: np.abs(sv).mean(axis=0) for k, (X, sv) in shap_store.items()}, index=FEATS)
sign = pd.DataFrame({k: [np.corrcoef(X[c].fillna(X[c].median()), sv[:, j])[0, 1] for j, c in enumerate(FEATS)] for k, (X, sv) in shap_store.items()}, index=FEATS)
stab = imp.rank().corr(method="spearman").values[np.triu_indices(len(imp.columns), 1)].mean()
shap_tab = pd.DataFrame({"mean_abs_shap": imp.mean(axis=1), "rank_mean": imp.rank(ascending=False).mean(axis=1),
                         "sign_corr_mean": sign.mean(axis=1), "sign_consistent_share": (np.sign(sign).eq(np.sign(sign.mean(axis=1)), axis=0)).mean(axis=1)}).sort_values("mean_abs_shap", ascending=False)
shap_tab.round(5).to_csv(TAB / "04_shap.csv")
print(f"mean pairwise Spearman of importance ranks across folds: {stab:.3f}")
X_all = pd.concat([X for X, _ in shap_store.values()]); sv_all = np.vstack([sv for _, sv in shap_store.values()])
plt.figure()
shap.summary_plot(sv_all, X_all, show=False, max_display=14)
plt.title("Q: which inputs drive LightGBM, and in which direction?", fontsize=10)
plt.tight_layout(); plt.savefig(FIG / "04_shap_beeswarm.png", dpi=150, bbox_inches="tight"); plt.close()
shap_tab.round(4)

mean pairwise Spearman of importance ranks across folds: 0.529


,mean_abs_shap,rank_mean,sign_corr_mean,sign_consistent_share
mkt_rvol_pct,0.0010,2.4444,-0.6027,0.8889
mom_252_21,0.0009,3.7778,0.7377,1.0000
sharpe_126,0.0008,3.7778,-0.6056,1.0000
sma_ratio_200,0.0006,6.0000,0.6052,0.8889
mom_63,0.0006,7.1111,0.7306,1.0000
mom_accel,0.0006,5.2222,-0.4312,0.8889
mkt_trend,0.0005,6.4444,-0.1830,0.6667
corr_spy_tlt_63,0.0005,7.6667,-0.3825,0.8889
asset_rvol_pct,0.0004,7.7778,0.6423,1.0000
skew_63,0.0004,8.5556,-0.8077,1.0000


## Conformal prediction intervals: empirical coverage per fold

In [9]:
conf = pd.DataFrame(conformal_rows)
conf["flag_<80%"] = conf.coverage < 0.8
conf.round(3).to_csv(TAB / "04_conformal.csv", index=False)
conf.pivot_table(index="fold", columns=["model", "level"], values="coverage").round(3)

model   lgbm         ridge       
level    0.8    0.9    0.8    0.9
fold                             
2015   0.753  0.863  0.754  0.863
2016   0.804  0.893  0.809  0.896
2017   0.881  0.953  0.879  0.952
2018   0.718  0.855  0.720  0.856
2019   0.811  0.915  0.815  0.914
2020   0.634  0.758  0.634  0.761
2021   0.876  0.949  0.877  0.948
2022   0.706  0.822  0.704  0.825
2023   0.893  0.952  0.898  0.953

In [10]:
pd.DataFrame(ridge_coefs).round(5).to_csv(TAB / "04_ridge_coefs.csv")
pd.DataFrame(ridge_coefs).mean(axis=1).sort_values().round(5)

mom_252_21_x_vol             -0.00038
skew_63                      -0.00031
sharpe_126_x_vol             -0.00024
mom_126_21_x_vol             -0.00019
skew_63_x_vol                -0.00015
sma_ratio_200_x_vol          -0.00007
mom_accel_x_vol              -0.00006
mom_accel                    -0.00006
sharpe_126                   -0.00004
trend_consistency_63_x_vol   -0.00001
mom_63_x_vol                  0.00002
amihud_63                     0.00017
mom_126_21                    0.00019
sma_ratio_200                 0.00020
mom_252_21                    0.00021
mom_63                        0.00028
amihud_63_x_vol               0.00029
trend_consistency_63          0.00030
dtype: float64